# ⚽ Football Pass Detection — Google Colab

Notebook này chạy pipeline phân tích chuyền bóng trên **GPU T4 miễn phí** của Colab.

**Trước khi chạy:** Vào `Runtime → Change runtime type → GPU (T4)`

## 1️⃣ Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ Clone repo & cài đặt dependencies

In [ ]:
# Clone repository
!git clone https://github.com/khangnguyenthe18/football_analysis.git
%cd football_analysis

In [ ]:
# Cài đặt package sports + dependencies
!pip install -e .
!pip install ultralytics gdown pandas

## 3️⃣ Tải model weights & video mẫu từ Google Drive

In [ ]:
import os
DATA_DIR = "examples/football/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Tải 3 model weights
!gdown -O "{DATA_DIR}/football-ball-detection.pt" "https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V"
!gdown -O "{DATA_DIR}/football-player-detection.pt" "https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q"
!gdown -O "{DATA_DIR}/football-pitch-detection.pt" "https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf"

# Tải 1 video mẫu (08fd33_0.mp4 ~ 20MB)
!gdown -O "{DATA_DIR}/08fd33_0.mp4" "https://drive.google.com/uc?id=1OG8K6wqUw9t7lp9ms1M48DxRhwTYciK-"

print("\n✅ Tải xong! Danh sách file:")
!ls -lh {DATA_DIR}/

## 4️⃣ Chạy Pass Detection với GPU 🚀

Lệnh dưới đây sẽ:
- Nhận diện cầu thủ + phân loại đội bóng
- Phát hiện bóng + theo dõi vị trí
- Phát hiện sân + tính homography
- Chạy FSM nhận diện chuyền bóng
- Xuất video kết quả + bảng thống kê

⏱️ **Thời gian dự kiến: ~15-30 phút trên T4 GPU** (so với 5-6 tiếng trên CPU)

In [ ]:
# Chạy pass detection - lưu ý: --device cuda để dùng GPU
!python examples/football/main.py \
    --source_video_path examples/football/data/08fd33_0.mp4 \
    --target_video_path examples/football/data/08fd33_0-pass-detection.mp4 \
    --device cuda \
    --mode PASS_DETECTION

## 5️⃣ Xem kết quả video

In [ ]:
import os
import subprocess
from IPython.display import HTML
from base64 import b64encode

raw_video = "examples/football/data/08fd33_0-pass-detection.mp4"
web_video = "examples/football/data/08fd33_0-pass-detection-web.mp4"

if os.path.exists(raw_video):
    # Nén & chuyển đổi sang H.264 chuẩn HTML5 (giảm từ ~70MB xuống ~8MB để phát trực tiếp mượt mà)
    print("⏳ Đang chuyển đổi video sang chuẩn H.264 để phát trực tiếp...")
    subprocess.run([
        "ffmpeg", "-y", "-i", raw_video,
        "-vcodec", "libx264", "-pix_fmt", "yuv420p",
        "-crf", "23", "-preset", "fast",
        web_video
    ], capture_output=True)

    target_file = web_video if os.path.exists(web_video) else raw_video
    file_size_mb = os.path.getsize(target_file) / (1024 * 1024)
    print(f"✅ Video H.264 sẵn sàng: {target_file} ({file_size_mb:.1f} MB)")

    mp4 = open(target_file, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=850 controls autoplay muted style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.5);">
        <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("❌ Video chưa được tạo. Hãy chạy Cell 4 (bước 4️⃣) trước.")

## 6️⃣ Tải video về máy tính

In [ ]:
from google.colab import files

download_file = web_video if 'web_video' in locals() and os.path.exists(web_video) else raw_video
if os.path.exists(download_file):
    print(f"✅ Đang tải {download_file} về máy...")
    files.download(download_file)
else:
    print("❌ File không tồn tại.")

## 7️⃣ (Tùy chọn) Chạy các mode khác

Bạn có thể thay `--mode` bằng các giá trị sau:
- `PITCH_DETECTION` — Phát hiện đường vạch sân
- `PLAYER_DETECTION` — Nhận diện cầu thủ
- `BALL_DETECTION` — Phát hiện bóng
- `PLAYER_TRACKING` — Theo dõi cầu thủ
- `TEAM_CLASSIFICATION` — Phân loại đội bóng
- `RADAR` — Bản đồ chiến thuật 2D
- `PASS_DETECTION` — Nhận diện chuyền bóng (đã chạy ở trên)

In [ ]:
# Ví dụ: chạy RADAR mode
# !python examples/football/main.py \
#     --source_video_path examples/football/data/08fd33_0.mp4 \
#     --target_video_path examples/football/data/08fd33_0-radar.mp4 \
#     --device cuda \
#     --mode RADAR